# Phase 5: Analysis & Ablations

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_DIR, 'data')):
    REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)

TRAIN = {l1: os.path.join(REPO_DIR, f'data/train/{l1}/kvl_shared_task_{l1}_train.csv') for l1 in ['es','de','cn']}
DEV = {l1: os.path.join(REPO_DIR, f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv') for l1 in ['es','de','cn']}
FIG_DIR = os.path.join(REPO_DIR, 'results/figures')
os.makedirs(FIG_DIR, exist_ok=True)

## Analysis 1 — Ablation Table

In [ ]:
# Remove one feature group at a time: all, -cognate, -frequency, -clue/orthographic, -semantic, -context
# 6 conditions x 3 L1s. Record RMSE per run.
from sklearn.metrics import mean_squared_error
from features.pipeline import FeaturePipeline
from models.xgboost_baseline import XGBoostBaseline

ablation_groups = {
    'all': None,
    '-cognate': ['edit_distance','char_ngram_overlap','is_cognate','length_ratio','l1_is_compound','cn_stroke_complexity'],
    '-frequency': ['subtlex_freq','aoa_score','concreteness','imageability'],
    '-clue_ortho': ['reveal_ratio','hidden_chars','word_length','syllable_count','has_prefix','has_suffix','letter_frequency_score'],
    '-semantic': ['wordnet_num_senses','wordnet_depth','cefr_level','pos_NOUN','pos_VERB','pos_ADJ','pos_ADV'],
    '-context': ['context_length','context_ttr','target_position_ratio'],
}
results = []
for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    dev_df = pd.read_csv(DEV[l1])
    for cond, drop in ablation_groups.items():
        pipe = FeaturePipeline(l1=l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
        pipe.fit(train_df)
        X_tr = pipe.transform(train_df)
        all_feats = [c for c in X_tr.columns if c != 'GLMM_score']
        if drop:
            pipe._frozen_names = [c for c in all_feats if c not in drop]
            X_tr = pipe.transform(train_df)
        else:
            pipe._frozen_names = None
        if 'GLMM_score' not in X_tr.columns: continue
        feat_cols = [c for c in X_tr.columns if c != 'GLMM_score']
        from xgboost import XGBRegressor
        model = XGBRegressor(n_estimators=200, max_depth=6, random_state=42)
        model.fit(X_tr[feat_cols], X_tr['GLMM_score'])
        X_dev = pipe.transform(dev_df)
        pred = model.predict(X_dev[[c for c in feat_cols if c in X_dev.columns]].fillna(0))
        rmse = np.sqrt(mean_squared_error(dev_df['GLMM_score'], pred))
        results.append({'L1': l1, 'condition': cond, 'RMSE': rmse})
ablation_df = pd.DataFrame(results)
print(ablation_df.pivot(index='condition', columns='L1', values='RMSE').round(4))

## Analysis 2 — SHAP on Final Hybrid Model

In [ ]:
# SHAP on MLP head (feature inputs). Compare with Phase 3 XGBoost SHAP.
print('Use SHAP on the concatenated feature vector to the MLP; compare with XGBoost SHAP from Phase 3.')

## Analysis 3 — Error Analysis

In [ ]:
# Top 50 errors per L1. Categorize false-easy vs false-hard. 3 examples per L1.
pred_path = os.path.join(REPO_DIR, 'results/predictions')
for l1 in ['es','de','cn']:
    dev_df = pd.read_csv(DEV[l1])
    pred_file = os.path.join(pred_path, f'exp4_hybrid_{l1}_seed42_dev.csv')
    if not os.path.isfile(pred_file):
        print(f'No pred file for {l1}')
        continue
    pred_df = pd.read_csv(pred_file)
    merged = dev_df.merge(pred_df, on='item_id')
    merged['error'] = merged['prediction'] - merged['GLMM_score']
    merged['abs_error'] = np.abs(merged['error'])
    top50 = merged.nlargest(50, 'abs_error')
    false_easy = top50[top50['error'] > 0]  # predicted easier (higher) than gold
    false_hard = top50[top50['error'] < 0]
    print(f'L1={l1} False-easy examples:')
    print(false_easy[['en_target_word','GLMM_score','prediction','error']].head(3))
    print('False-hard examples:')
    print(false_hard[['en_target_word','GLMM_score','prediction','error']].head(3))

## Analysis 4 — Cross-L1 Transfer

In [ ]:
# Train XGBoost on one L1, eval on all. Test if difficulty is L1-specific.
from features.pipeline import FeaturePipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

transfer = []
for train_l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[train_l1])
    pipe = FeaturePipeline(l1=train_l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipe.fit(train_df)
    X_tr = pipe.transform(train_df)
    feat_cols = [c for c in X_tr.columns if c != 'GLMM_score']
    model = XGBRegressor(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X_tr[feat_cols], X_tr['GLMM_score'])
    for eval_l1 in ['es','de','cn']:
        dev_df = pd.read_csv(DEV[eval_l1])
        pipe_eval = FeaturePipeline(l1=eval_l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
        pipe_eval.fit(pd.read_csv(TRAIN[eval_l1]))
        X_dev = pipe_eval.transform(dev_df)
        cols = [c for c in feat_cols if c in X_dev.columns]
        X_dev = X_dev[cols].reindex(columns=feat_cols).fillna(0)
        pred = model.predict(X_dev)
        rmse = mean_squared_error(dev_df['GLMM_score'], pred, squared=False)
        transfer.append({'train_L1': train_l1, 'eval_L1': eval_l1, 'RMSE': rmse})
print(pd.DataFrame(transfer).pivot(index='train_L1', columns='eval_L1', values='RMSE').round(4))

## Analysis 5 — Difficulty Clustering

In [ ]:
from sklearn.cluster import KMeans

dev_es = pd.read_csv(DEV['es'])[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_es'})
dev_de = pd.read_csv(DEV['de'])[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_de'})
dev_cn = pd.read_csv(DEV['cn'])[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_cn'})
m = dev_es.merge(dev_de, on='item_id').merge(dev_cn, on='item_id')
X = m[['GLMM_es','GLMM_de','GLMM_cn']].values
for k in [3, 4]:
    km = KMeans(n_clusters=k, random_state=42)
    m[f'cluster_k{k}'] = km.fit_predict(X)
    print(f'k={k} centers:', km.cluster_centers_)
plt.style.use('seaborn-v0_8-paper')
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(m['GLMM_es'], m['GLMM_de'], m['GLMM_cn'], c=m['cluster_k3'], cmap='Set1')
ax.set_xlabel('GLMM_es'); ax.set_ylabel('GLMM_de'); ax.set_zlabel('GLMM_cn')
plt.title('Difficulty clusters (k=3)')
plt.savefig(os.path.join(FIG_DIR, 'difficulty_clusters.pdf'), bbox_inches='tight')
plt.show()

## Analysis 6 — Publication-Ready Figures

In [ ]:
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
# ColorBrewer palettes, 10pt labels, PDF output, white background.
print('Use plt.style.use(\'seaborn-v0_8-paper\'), ColorBrewer, 10pt labels, PDF.')